# 01 — Baseline op_03 (validation temporelle, anti-fuite)

Baseline **honnête** qui suit `FINDINGS.md` :
1. Restriction à `op_03` (toute la fraude y est, proba 0 ailleurs).
2. Features comportementales **fold-safe** (pas d'encodage d'ID brut, pas de fuite).
3. **Validation temporelle** (`time_folds`) — la CV aléatoire ment ici.
4. Calibration isotonic.
5. Soumission au format `id,target`.

> Ce notebook = exploration. Dès qu'une fonction se stabilise, elle part dans `src/`.

## Setup
Si l'import échoue : `pip install -r ../requirements.txt` dans ton environnement.

In [ ]:
! pip install -r ../requirements.txt

In [ ]:
# recharge automatiquement les modules src/ quand ils changent (évite de redémarrer le kernel)
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# rendre le package src/ importable depuis notebooks/
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src import config as C
from src.validation import time_folds, stratified_folds, evaluate_ap
from src.utils import op03_mask, assemble_full_proba, make_submission, seed_everything
from src.calibration import fit_isotonic, apply_isotonic

seed_everything(42)
DATA = ROOT / "data"
pd.set_option("display.max_columns", 50)

In [ ]:
train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
sample = pd.read_csv(DATA / "sample_submission.csv")
print("train:", train.shape, "| test:", test.shape)
train.head()

## Vérification des findings EDA

In [ ]:
# 1) Toute la fraude est-elle dans op_03 ?
fraud_by_op = train.groupby(C.OPERATION)[C.TARGET].agg(["mean", "sum", "count"])
print(fraud_by_op)

# 2) Frontières temporelles train vs test
print("\ntrain period:", train[C.PERIOD].min(), "->", train[C.PERIOD].max())
print("test  period:", test[C.PERIOD].min(), "->", test[C.PERIOD].max())

# 3) Le taux de fraude varie-t-il dans le temps ?
rate = train.groupby(C.PERIOD)[C.TARGET].mean()
print("\ntaux de fraude par période: min %.3f / max %.3f" % (rate.min(), rate.max()))

## Restriction à op_03
On modélise uniquement op_03. Les autres opérations auront une proba forcée à 0.

In [ ]:
tr03 = train[op03_mask(train)].reset_index(drop=True)
te03 = test[op03_mask(test)].reset_index(drop=True)
y = tr03[C.TARGET].values
print("op_03 train:", tr03.shape, "| taux fraude:", round(y.mean(), 4))
print("op_03 test :", te03.shape)

## Features de base (row-level, sans fuite)
Transformations ligne par ligne uniquement — pas d'agrégation globale ici.
Les fréquences fold-safe sont ajoutées dans la boucle de CV (cellule suivante).

In [ ]:
from src.features.temporal import balance_features
EPS = 1e-6

def base_features(df, with_balance=True):
    f = pd.DataFrame(index=df.index)
    f["amount_log1p"] = np.log1p(np.maximum(df[C.AMOUNT], 0))
    f["amount_vs_origin_before"] = df[C.AMOUNT] / (np.abs(df[C.ORIGIN_BAL_BEFORE]) + EPS)
    f["amount_vs_dest_before"] = df[C.AMOUNT] / (np.abs(df[C.DEST_BAL_BEFORE]) + EPS)
    f["origin_balance_before"] = df[C.ORIGIN_BAL_BEFORE]
    f["dest_balance_before"] = df[C.DEST_BAL_BEFORE]
    if with_balance:
        # DÉCOUVERTE EDA (notebook 00) : l'incohérence de solde est le signal
        # univarié le plus fort dans op_03. On l'ajoute et on mesure le gain.
        f = pd.concat([f, balance_features(df)], axis=1)
    return f

Xtr_base = base_features(tr03)
Xte_base = base_features(te03)
print("features:", list(Xtr_base.columns))
Xtr_base.head()

## Validation temporelle + fréquences fold-safe
Pour chaque fold, les fréquences de comptes sont apprises **uniquement sur le passé**
(les lignes d'entraînement du fold), jamais sur la validation. C'est l'anti-fuite.

In [ ]:
def make_model():
    """CatBoost si dispo (modèle de référence), sinon repli sklearn."""
    try:
        from catboost import CatBoostClassifier
        m = CatBoostClassifier(loss_function="Logloss", eval_metric="PRAUC",
                               depth=6, learning_rate=0.05, iterations=600,
                               random_seed=42, verbose=False)
        return m, "catboost"
    except ImportError:
        from sklearn.ensemble import HistGradientBoostingClassifier
        m = HistGradientBoostingClassifier(max_iter=400, learning_rate=0.05,
                                           random_state=42)
        return m, "sklearn-histgbm"

FREQ_COLS = [C.ORIGIN_ACCT, C.DEST_ACCT]

def add_freq(X, src_df, ref_df):
    """Ajoute des fréquences de comptes apprises sur ref_df, appliquées à src_df."""
    X = X.copy()
    for col in FREQ_COLS:
        freq = ref_df[col].value_counts(normalize=True)
        X[f"freq_{col}"] = src_df[col].map(freq).fillna(0).values
    return X

folds = list(time_folds(tr03[C.PERIOD]))
oof = np.zeros(len(y))
model_name = None
for k, (tr_idx, va_idx) in enumerate(folds):
    # fréquences apprises sur le PASSÉ (tr_idx) uniquement
    X_tr = add_freq(Xtr_base.iloc[tr_idx], tr03.iloc[tr_idx], tr03.iloc[tr_idx])
    X_va = add_freq(Xtr_base.iloc[va_idx], tr03.iloc[va_idx], tr03.iloc[tr_idx])
    model, model_name = make_model()
    model.fit(X_tr, y[tr_idx])
    oof[va_idx] = model.predict_proba(X_va)[:, 1]
    print(f"fold {k}: AP = {evaluate_ap(y[va_idx], oof[va_idx]):.4f}")

print(f"\nModèle: {model_name}")
print(f"AP OOF (validation TEMPORELLE) = {evaluate_ap(y, oof):.4f}")

In [ ]:
# A/B : gain réel des features d'incohérence de solde (même CV temporelle)
Xtr_nobal = base_features(tr03, with_balance=False)
oof_nobal = np.zeros(len(y))
for tr_idx, va_idx in folds:
    Xa = add_freq(Xtr_nobal.iloc[tr_idx], tr03.iloc[tr_idx], tr03.iloc[tr_idx])
    Xb = add_freq(Xtr_nobal.iloc[va_idx], tr03.iloc[va_idx], tr03.iloc[tr_idx])
    m, _ = make_model()
    m.fit(Xa, y[tr_idx])
    oof_nobal[va_idx] = m.predict_proba(Xb)[:, 1]

ap_with = evaluate_ap(y, oof)
ap_without = evaluate_ap(y, oof_nobal)
print(f"AP SANS features de solde : {ap_without:.4f}")
print(f"AP AVEC features de solde : {ap_with:.4f}")
print(f"Gain global                : {ap_with - ap_without:+.4f}")
# Détail par fold (le fold le plus récent = meilleur proxy du test)
for k, (_, va) in enumerate(folds):
    print(f"  fold {k}: {evaluate_ap(y[va], oof_nobal[va]):.4f} -> {evaluate_ap(y[va], oof[va]):.4f}")

### Pourquoi la validation aléatoire ment
Même features, même modèle, mais folds aléatoires : le score est gonflé et trompeur.
C'est exactement le piège que le rapport signale.

In [ ]:
oof_rand = np.zeros(len(y))
for tr_idx, va_idx in stratified_folds(pd.Series(y)):
    X_tr = add_freq(Xtr_base.iloc[tr_idx], tr03.iloc[tr_idx], tr03.iloc[tr_idx])
    X_va = add_freq(Xtr_base.iloc[va_idx], tr03.iloc[va_idx], tr03.iloc[tr_idx])
    m, _ = make_model()
    m.fit(X_tr, y[tr_idx])
    oof_rand[va_idx] = m.predict_proba(X_va)[:, 1]

print(f"AP CV aléatoire (TROMPEUSE) = {evaluate_ap(y, oof_rand):.4f}")
print(f"AP CV temporelle (FIABLE)   = {evaluate_ap(y, oof):.4f}")

## Calibration isotonic (sur l'out-of-fold)

In [ ]:
iso = fit_isotonic(oof, y)
oof_cal = apply_isotonic(iso, oof)
print(f"AP OOF avant calib : {evaluate_ap(y, oof):.4f}")
print(f"AP OOF après calib : {evaluate_ap(y, oof_cal):.4f}")
print("Note: la calibration isotonic crée des paquets d'ex-aequo (zones plates) ;")
print("l'AP peut donc bouger légèrement. La calibration vise la QUALITÉ des probas")
print("(exigée par la brief), pas l'AP. On ne juge jamais un modèle sur l'AP post-calib.")

## Modèle final + soumission
Réentraînement sur tout le train op_03 (fréquences sur tout le train), prédiction du test op_03,
puis proba 0 forcée hors op_03.

In [ ]:
X_full = add_freq(Xtr_base, tr03, tr03)
X_test = add_freq(Xte_base, te03, tr03)
final, _ = make_model()
final.fit(X_full, y)
proba03 = apply_isotonic(iso, final.predict_proba(X_test)[:, 1])

# reconstruire le vecteur complet : proba sur op_03, 0 ailleurs
full = np.zeros(len(test))
full[op03_mask(test).values] = proba03

path = make_submission(test[C.ID], full, "01_baseline_op03")
print("soumission écrite :", path)

# contrôles format
sub = pd.read_csv(path)
assert list(sub.columns) == ["id", "target"]
assert len(sub) == len(test)
assert set(sub["id"]) == set(sample["id"])
assert sub["target"].between(0, 1).all()
print("format OK — proba non nulles:", int((sub['target'] > 0).sum()), "(= nb d'op_03 dans le test)")